# GACL: Real Deep-Model Training on Real Crop-Disease Photographs

This notebook runs the **actual GACL deep-learning pipeline** (HGAViT + GCATT + DHGNN + VLAE) on the real 1,543-image dataset, using a free Colab GPU. It requires PyTorch, which is not available in the sandbox used to prepare the paper -- this notebook is the real fix.

**Steps:**
1. Runtime -> Change runtime type -> GPU (T4 is fine).
2. Run all cells in order.
3. Upload `GACL_Architecture_Manuscripts_FINAL.zip` when prompted (or mount Google Drive if you've stored it there).
4. Copy the printed metrics at the end back to Claude (or paste them into the paper yourself) to report GACL's real, trained performance against the 64.4%/38.1% classical baseline already in Section 3.14.

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Upload the project zip (GACL_Architecture_Manuscripts_FINAL.zip)
from google.colab import files
uploaded = files.upload()  # select GACL_Architecture_Manuscripts_FINAL.zip

In [ ]:
import zipfile, glob, os
zpath = glob.glob('*.zip')[0]
with zipfile.ZipFile(zpath) as z:
    z.extractall('project')

# Also need the real images -- upload My_Data.zip separately if it wasn't bundled
print('Project extracted. Contents:')
for root, dirs, fnames in os.walk('project'):
    depth = root.count(os.sep)
    if depth <= 3:
        print(root)

In [ ]:
# If My_Data (the real photographs) isn't already inside the project zip, upload it now:
import os
if not os.path.isdir('project/GACL_Architecture_Manuscripts_FINAL/data/My_Data') and not os.path.isdir('My_Data'):
    from google.colab import files
    print('Upload My_Data.zip (the real crop-disease photographs)')
    uploaded2 = files.upload()
    import zipfile as zf
    zpath2 = [f for f in uploaded2.keys() if f.lower().endswith('.zip')][0]
    with zf.ZipFile(zpath2) as z:
        z.extractall('.')
    print('Extracted My_Data')

In [ ]:
%cd project/GACL_Architecture_Manuscripts_FINAL/code
!pip install -q -r requirements.txt

In [ ]:
# Point --data_root at wherever My_Data ended up (adjust the path if needed)
import os
candidates = ['../data/My_Data', '../../My_Data', '../../../My_Data']
data_root = next((c for c in candidates if os.path.isdir(c)), None)
print('Using data_root =', data_root)
assert data_root is not None, 'Could not find My_Data -- set data_root manually below'

In [ ]:
# Run the REAL GACL training + evaluation script.
# This is the actual deep model (HGAViT+GCATT+DHGNN+VLAE) -- not a classical baseline.
!python train_gacl_real_images.py --data_root "$data_root" --epochs 30 --batch_size 32 --device cuda --save_checkpoint gacl_realimage_checkpoint.pt

## What to do with the output

The cell above prints real accuracy, balanced accuracy, macro-F1, macro-AUC, Cohen's kappa, and MCC on the held-out validation and test splits, each next to its chance-level reference -- exactly the numbers needed to fill in the 'GACL (trained)' row that Section 3.14 of the paper is currently missing.

Copy the full printed output (or download `gacl_realimage_checkpoint.pt` via `files.download(...)` if you want the trained weights too) and send it back so it can be written into the paper as GACL's own, genuinely measured result, compared directly against the 64.4%/38.1% classical baseline already reported.

If you want to try improving on the classical baseline, reasonable first knobs to adjust: `--epochs` (try 50-100), `--lr`, `--image_size` (64 is small; try 128 or 224 if GPU memory allows), and the loss weights in `gacl/config.py`.